In [78]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from dataclasses import dataclass

load_dotenv()

True

In [79]:
@dataclass(frozen=True)
class Provider:
    """Provider class to return the available provider based on the environment variable."""

    name: str
    env_var: str
    base_url: str
    model: str

In [80]:
PROVIDERS = [
    Provider(
        name="Open Router",
        env_var="OPEN_ROUTER_API_KEY",
        base_url="https://openrouter.ai/api/v1",
        model="openai/gpt-4o"
    ),
    Provider(
        name="Gemini",
        env_var="GEMINI_API_KEY",
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        model="gemini-3.5-flash"
    ),
    Provider(
        name="Groq",
        env_var="GROQ_API_KEY",
        base_url="https://api.groq.com/openai/v1",
        model="openai/gpt-oss-20b"
    )
]

In [81]:
def select_provider() -> Provider:
    """Select the provider based on the environment variable."""
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    raise ValueError("No valid provider found. Please set the appropriate environment variable.")

In [82]:
def build_client(provider: Provider) -> OpenAI:
    """Build the OpenAI client based on the selected provider."""
    api_key = os.getenv(provider.env_var)
    if not api_key:
        raise ValueError(f"API key for {provider.name} is not set in environment variables.")
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url
    )

In [83]:
provider = select_provider()
client = build_client(provider)

In [84]:
SYSTEM_PROMPT = """
    You are a mathematics assistant.
    You will receive a maths problem in simple string.
    You have to solve that problem with chain of thought problem
    You have to solve in step by step way
    After the complete solution you have to return answer in below format:-
        Final Answer: <numerical_answer>
    After Final Answer: <numerical_answer> dont add any text or anything
"""

In [85]:
def llm_reply(prompt, temperature=1):
    response = client.chat.completions.create(
        model=provider.model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [86]:
def extract_final_number(answer: str | None) -> float:
    if not answer or not answer.split():
        raise ValueError("The response does not contain a final answer.")

    number_text = answer.split()[-1]
    number = float(number_text)
    return number

In [87]:
PROMPT = """
    Mark earns $15 an hour mowing lawns and $10 an hour raking leaves. He mowed lawns for 4 hours and raked leaves for 3 hours last week. How much total money did Mark earn?
    Let's solve this step by step.
"""

In [88]:
raw_response = llm_reply(PROMPT)
numeric_answer = extract_final_number(raw_response)
print(numeric_answer)

90.0
